Step 1: Preprocessing meteostat data

In [1]:
from meteostat import Point, Hourly
from datetime import datetime
import pandas as pd
import numpy as np

Set location and time param

In [2]:
# Parameters
lat, lon = 34.0008, -81.0351
start = datetime(2024,1,1)
end = datetime(2025,1,1)
city = Point(lat, lon)

Get data

In [3]:
# Fetch data from api
df = Hourly(city, start, end).fetch()
print(df)

# ensure timestamps are datetime
df.index = pd.to_datetime(df.index)

# Select & rename columns
df = df[['temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'pres', 'coco']]

# Ensure hourly continuity
full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H', tz=df.index.tz)
df = df.reindex(full_idx)

                     temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt    pres  \
time                                                                           
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  <NA>  210.0   6.0  <NA>  1020.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  <NA>  210.0   5.4  <NA>  1020.2   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0  <NA>    0.0   0.0  <NA>  1020.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0  <NA>    0.0   0.0  <NA>  1019.9   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0  <NA>    0.0   0.0  <NA>  1019.9   
...                   ...   ...   ...   ...   ...    ...   ...   ...     ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  <NA>  240.0  20.5  <NA>  1004.7   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  <NA>  230.0  16.6  <NA>  1004.5   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  <NA>  230.0  20.5  <NA>  1004.9   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  <NA>  210.0  11.2  <NA>  1005.2   
2025-01-01 00:00:00  18.0  11.3  65.0   

Fill in precips with NaN = 0 and log transform, add boolean rain flag

In [4]:
# assume NaN prcp = 0
df['prcp'] = df['prcp'].fillna(0)
# Change precip to either 0 or log-transformed amount
df['prcp'] = np.log1p(df['prcp'])
# add a rain flag (0 or 1)
df['prcp_flag'] = (df['prcp'] > 0).astype(int)
print(df)

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag  
2024-01-01 00:00:00          0  
2024-01-01 01:00:00          0  
2024-01-01 0

If timestamps missing (unlikely) fill them in by interpolating small gaps and forward fill long gaps

In [5]:
# Impute missing timestamps
# short gaps (<=3h): linear; long gaps: forward fill and add mask
gap_mask = df.isna().any(axis=1)
print(gap_mask[1].sum())
df_short = df.interpolate(limit=3, limit_direction='both')
print(df_short)
# long gaps
df_long = df_short.fillna(method='ffill')
print(df_long)

0
                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag  
2024-01-01 00:00:00          0  
2024-01-01 01:00:00          0  
2024-01-01

Standardize variables and clip outliers

In [6]:
# Standardize numerical units and clip outliers (3 sigma)
df_std= df_long.copy()
for col in ['temp', 'dwpt', 'rhum', 'wspd', 'pres']:
    var = df_long[col]
    mu, sigma = var.mean(), var.std()
    # clip values greater than 3 stds
    clipped = var.clip(lower=mu - 3*sigma, upper=mu + 3*sigma)

print(df_std)

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag  
2024-01-01 00:00:00          0  
2024-01-01 01:00:00          0  
2024-01-01 0

ecoding time using cyclical sin/cos transformation 

In [7]:
# Convert time to seconds
timestamp_s = df_std.index.map(pd.Timestamp.timestamp)
print(timestamp_s)

# Get the number of seconds for each time period
day = 24*60*60
week = day*7
year = day*(365.2425)

# Transform using sin and cos
# Time of day
df_std['day_sin'] = np.sin(timestamp_s * (2 * np.pi / day))
df_std['day_cos'] = np.cos(timestamp_s * (2 * np.pi / day))

# Time of week
df_std['week_sin'] = np.sin(timestamp_s * (2 * np.pi / week))
df_std['week_cos'] = np.cos(timestamp_s * (2 * np.pi / week))

# Time of year
df_std['year_sin'] = np.sin(timestamp_s * (2 * np.pi / year))
df_std['year_cos'] = np.cos(timestamp_s * (2 * np.pi / year))

print(df_std)

Index([1704067200.0, 1704070800.0, 1704074400.0, 1704078000.0, 1704081600.0,
       1704085200.0, 1704088800.0, 1704092400.0, 1704096000.0, 1704099600.0,
       ...
       1735657200.0, 1735660800.0, 1735664400.0, 1735668000.0, 1735671600.0,
       1735675200.0, 1735678800.0, 1735682400.0, 1735686000.0, 1735689600.0],
      dtype='float64', length=8785)
                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  

Encode wind speed as sin/cos too since it is circular

In [8]:
df_std['wdir_sin'] = np.sin(np.deg2rad(df['wdir']))
df_std['wdir_cos'] = np.cos(np.deg2rad(df['wdir']))
print(df_std)


                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag       day_sin   day_cos  week_sin  week_cos  \
2024-01-01 00:00:00          0 -1

See which features are most correlated to determine while lag variables to create
Don't want to create redundant lag variabels

In [9]:
df_std[['temp', 'rhum', 'dwpt', 'wspd', 'wdir', 'pres']].corr()

,temp,rhum,dwpt,wspd,wdir,pres
temp,1.000000,-0.166142,0.808068,0.201406,0.170260,-0.395497
rhum,-0.166142,1.000000,0.437173,-0.419677,-0.379338,-0.083270
dwpt,0.808068,0.437173,1.000000,-0.065642,-0.071292,-0.402702
wspd,0.201406,-0.419677,-0.065642,1.000000,0.689665,-0.335895
wdir,0.170260,-0.379338,-0.071292,0.689665,1.000000,-0.296155
pres,-0.395497,-0.083270,-0.402702,-0.335895,-0.296155,1.000000


Adding lag features (temp, pressure, humidity, wind speed)

In [10]:

# more important variables, longer lag
lag_hours = [1, 3, 6, 24]
vars_to_lag = ['temp', 'rhum', 'pres']

for var in vars_to_lag:
    for lag in lag_hours:
        df_std[f'{var}_lag{lag}'] = df_std[var].shift(lag)

# less important, shorter lag to reduce noise
# dew point correlates with 
lag_hours = [1, 3, 6]
vars_to_lag = ['wspd', 'prcp', 'dwpt', 'wdir_sin', 'wdir_cos']

for var in vars_to_lag:
    for lag in lag_hours:
        df_std[f'{var}_lag{lag}'] = df_std[var].shift(lag)

print(df_std)



                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag       day_sin  ...  prcp_lag6  dwpt_lag1  \
2024-01-01 00:00:00          0 -1.39

Rolling stats

In [11]:
# rolling average
roll_hours = [3, 6, 12]
vars_to_roll = ['temp', 'rhum', 'wspd', 'prcp', 'dwpt']

for var in vars_to_roll:
    for window in roll_hours:
        df_std[f'{var}_roll{window}'] = df_std[var].shift(1).rolling(window, min_periods=1).mean()

# do sum of precip instead of avg
for window in [3, 6, 12]:
    df_std[f'prcp_sum{window}'] = df_std['prcp'].shift(1).rolling(window, min_periods=1).sum()
print(df_std)

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-01 00:00:00   9.0   0.2  54.0   0.0  210.0   6.0  1020.0   1.0   
2024-01-01 01:00:00   8.9   0.1  54.0   0.0  210.0   5.4  1020.2   3.0   
2024-01-01 02:00:00   6.1   1.6  73.0   0.0    0.0   0.0  1020.0   3.0   
2024-01-01 03:00:00   5.6   1.1  73.0   0.0    0.0   0.0  1019.9   3.0   
2024-01-01 04:00:00   6.1   1.0  70.0   0.0    0.0   0.0  1019.9   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag       day_sin  ...  wspd_roll12  prcp_roll3  \
2024-01-01 00:00:00          0 -1

Drop NA's resulting from lag and rolling

In [12]:
# drop NA's resulting from lag
lag_roll_cols = [col for col in df_std.columns if 'lag' or 'roll' in col]
#print(df_std)

#print(df_std[lag_cols].isna().sum()) 
#print(df_std[lag_cols].head(30)) 

df_std = df_std.dropna(subset=lag_roll_cols).copy()
print(df_std)

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-02 00:00:00  10.0   1.1  54.0   0.0   20.0  14.8  1019.2   3.0   
2024-01-02 01:00:00   8.9  -2.1  46.0   0.0  350.0  13.0  1020.2   3.0   
2024-01-02 02:00:00   6.7  -3.8  47.0   0.0    2.0   7.0  1021.2   3.0   
2024-01-02 03:00:00   6.1  -3.8  49.0   0.0  350.0   7.6  1022.2   2.0   
2024-01-02 04:00:00   5.6  -4.3  49.0   0.0  350.0   7.6  1022.8   3.0   
...                   ...   ...   ...   ...    ...   ...     ...   ...   
2024-12-31 20:00:00  21.7  12.8  57.0   0.0  240.0  20.5  1004.7   3.0   
2024-12-31 21:00:00  22.2  11.6  51.0   0.0  230.0  16.6  1004.5   1.0   
2024-12-31 22:00:00  21.1  10.0  49.0   0.0  230.0  20.5  1004.9   2.0   
2024-12-31 23:00:00  18.9  10.7  59.0   0.0  210.0  11.2  1005.2   1.0   
2025-01-01 00:00:00  18.0  11.3  65.0   0.0  210.0  15.0  1006.0   2.0   

                     prcp_flag       day_sin  ...  wspd_roll12  prcp_roll3  \
2024-01-02 00:00:00          0 -9

In [13]:
# Save cleaned dataset
df_std.to_parquet('hourly_columbia_weather.parquet')